In [4]:
import os
import re
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 50)


def find_repo_root(start: Path) -> Path:
    # Walk up from the notebook's folder until we find the repo root, so the
    # notebook runs from src/ or from the repo root.
    for candidate in [start, *start.parents]:
        if (candidate / "sql files").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repo root (folder containing 'sql files/' and README.md).")


ROOT = find_repo_root(Path.cwd().resolve())
DATA_DIR = ROOT / "data"
LOOKUP_DIR = ROOT / "instacart dataset"
OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

# --- Analysis parameters (from the Stage 1 proposal) ---
MIN_PURCHASES = 500          # eligibility floor
STABILITY_MIN = 1_000        # stricter floor for the stability check
TOP_N = 20                   # slots in the "Buy It Again" pilot
CAP_PER_DEPT = 2             # max products per department in the capped list
EXCLUDED_DEPARTMENT_ID = 21  # 'missing'
EXCLUDED_AISLE_ID = 100      # 'missing'

# --- Chart style (validated categorical slots 1-2; neutral inks for text/reference lines) ---
BLUE, ORANGE = "#257de0", "#f47230"
OTHER_GRAY = "#b0b0b0"
INK, INK_2, GRID = "#191919", "#666666", "#e6e6e6"
sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": GRID, "grid.color": GRID, "axes.labelcolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK, "text.color": INK,
    "axes.titleweight": "bold", "axes.titlesize": 13, "figure.dpi": 110, "savefig.dpi": 200,
})


def save_table(df: pd.DataFrame, name: str) -> None:
    df.to_csv(OUTPUTS / name, index=False)
    print(f"saved outputs/{name}  ({len(df):,} rows)")


def save_fig(fig, name: str) -> None:
    fig.savefig(OUTPUTS / name, bbox_inches="tight", facecolor="white")
    print(f"saved outputs/{name}")


print("Repo root folder:", ROOT.name)
print("pandas", pd.__version__, "| matplotlib", matplotlib.__version__, "| seaborn", sns.__version__)

Repo root folder: instacart-buy-it-again
pandas 3.1.1 | matplotlib 3.11.2 | seaborn 0.13.2
